# Competitive Pricing Intelligence & Simulator
## Phase 1 — Pricing Simulator MVP

**Business Problem**

PowerUp（虛構電商賣家）的行動電源和市場上多個競品（Anker、小米、ROMOSS、PhoneMax 等）規格接近、消費者容易比價。當我們自己的成本或市場條件改變時，我們應該如何定價，才能讓利潤最大化，而不只是追求銷量？

本篇 Notebook 建立一個最小可行的定價模擬模型（不含競品爬蟲，純粹是我們自己的 demand-cost-profit 模型），回答：

> 如果我們改變產品售價，需求量、營收與利潤會如何變化？最佳價格在哪裡？

**Base Assumptions**

| 參數 | 數值 | 說明 |
|---|---|---|
| Base price (P0) | 690 | 目前售價（假設） |
| Base demand (Q0) | 1000 units/月 | 假設值，非真實銷售資料 |
| Unit cost | 400 | 約 58% 成本率 |
| Price elasticity | -1.8 | 假設值：商品規格同質化、替代品多，需求對價格敏感（elastic demand） |

> ⚠️ **重要聲明**：以上參數均為假設值，用於展示分析方法，並非真實市場數據。本專案為 portfolio project，模型的目的是展示「定價決策的分析框架」，而非提供可直接用於生產環境的精準預測。


## 1. Demand Model（需求模型）

採用 **Constant Elasticity Demand Model**：

$$Q(P) = Q_0 \times \left(\frac{P}{P_0}\right)^{\varepsilon}$$

**為什麼用這個模型？**

1. 彈性（elasticity）在整條需求曲線上保持不變，直接對應經濟學課堂上「用彈性描述需求敏感度」的定義，不會像線性模型一樣彈性隨價格漂移。
2. 無論價格如何變動，需求量恆為正值，不會出現線性模型可能算出負需求的不合理情況。
3. 這是定價文獻中最常見的 price simulation 起手式模型。


In [1]:
def estimate_demand(price, base_price, base_demand, elasticity):
    """Constant elasticity demand model: Q(P) = Q0 * (P/P0)^elasticity"""
    return base_demand * (price / base_price) ** elasticity


# Base assumptions
P0 = 690        # base price
Q0 = 1000       # base demand
elasticity = -1.8
unit_cost = 400

# Sanity check
for p in [500, 600, 690, 750, 850]:
    print(f"Price {p}: estimated demand = {estimate_demand(p, P0, Q0, elasticity):.1f}")


Price 500: estimated demand = 1785.6
Price 600: estimated demand = 1286.0
Price 690: estimated demand = 1000.0
Price 750: estimated demand = 860.6
Price 850: estimated demand = 687.0


## 2. Revenue / Cost / Profit

$$Revenue = Price \times Quantity \qquad Total\ Cost = Unit\ Cost \times Quantity \qquad Profit = Revenue - Total\ Cost$$

我們先只考慮變動成本（Unit Cost × Quantity），因為固定成本不受定價策略影響，不會改變「最佳價格」的判斷。


In [2]:
def evaluate_price(price, base_price, base_demand, elasticity, unit_cost):
    demand = estimate_demand(price, base_price, base_demand, elasticity)
    revenue = price * demand
    total_cost = unit_cost * demand
    profit = revenue - total_cost
    margin = profit / revenue if revenue > 0 else 0
    return {
        "price": price, "demand": demand, "revenue": revenue,
        "total_cost": total_cost, "profit": profit, "margin": margin,
    }


import pandas as pd

sample_prices = [500, 600, 690, 750, 850]
sample_df = pd.DataFrame([evaluate_price(p, P0, Q0, elasticity, unit_cost) for p in sample_prices])
sample_df


,price,demand,revenue,total_cost,profit,margin
0,500,1785.592510,892796.254979,714237.003983,178559.250996,0.200000
1,600,1286.044844,771626.906516,514417.937677,257208.968839,0.333333
2,690,1000.000000,690000.000000,400000.000000,290000.000000,0.420290
3,750,860.633188,645474.890637,344253.275007,301221.615631,0.466667
4,850,687.027778,583973.611533,274811.111310,309162.500224,0.529412


## 3. 尋找 Profit-Maximizing Price

**方法一：Grid Search** — 掃過一段價格區間，逐一計算利潤，取最大值。

**方法二：經濟學公式解（Lerner Markup Rule）** — 對於 constant elasticity demand，利潤最大化的價格有解析解：

$$P^* = Unit\ Cost \times \frac{\varepsilon}{\varepsilon + 1}$$

這個公式來自「邊際收益 = 邊際成本」的最適化條件，直接對應經濟學的邊際分析。我們用它來驗證 grid search 的結果是否正確。


In [3]:
def find_optimal_price(base_price, base_demand, elasticity, unit_cost, price_min, price_max, step=5):
    results = []
    price = price_min
    while price <= price_max:
        results.append(evaluate_price(price, base_price, base_demand, elasticity, unit_cost))
        price += step
    best = max(results, key=lambda r: r["profit"])
    return best, results


# 理論公式解
theoretical_price = unit_cost * (elasticity / (elasticity + 1))
print(f"理論最佳價格（公式解）: {theoretical_price:.1f}")

# Grid search
best, all_results = find_optimal_price(P0, Q0, elasticity, unit_cost, price_min=400, price_max=1500, step=5)
results_df = pd.DataFrame(all_results)
print(f"Grid Search 找到的最佳價格: {best['price']}")
print(f"  demand={best['demand']:.1f}, revenue={best['revenue']:.0f}, profit={best['profit']:.0f}, margin={best['margin']:.1%}")


理論最佳價格（公式解）: 900.0
Grid Search 找到的最佳價格: 900
  demand=619.9, revenue=557872, profit=309929, margin=55.6%


**結果**：理論公式解與 Grid Search 結果一致（皆為 900），驗證程式邏輯正確。

**商業意義**：在目前假設下，公司目前定價（690）其實偏低。理論上調漲至 900 附近，雖然銷量會下滑，但總利潤與利潤率反而提升——這說明「銷量最大化」與「利潤最大化」並非同一件事。


## 4. 視覺化：Price vs Demand / Revenue / Profit

In [4]:
import plotly.express as px

def plot_price_vs_metric(df, metric, best_price, title):
    fig = px.line(df, x="price", y=metric, title=title)
    fig.add_vline(x=best_price, line_dash="dash", line_color="red",
                   annotation_text=f"Optimal Price = {best_price}")
    return fig

fig_demand = plot_price_vs_metric(results_df, "demand", best["price"], "Price vs Demand")
fig_demand.show()


In [5]:
fig_revenue = plot_price_vs_metric(results_df, "revenue", best["price"], "Price vs Revenue")
fig_revenue.show()


In [6]:
fig_profit = plot_price_vs_metric(results_df, "profit", best["price"], "Price vs Profit")
fig_profit.show()


**觀察**：Revenue 曲線在整個價格區間呈單調遞減——因為彈性假設在全區間都 > 1（elastic demand）的緣故。但 Profit 曲線出現明顯高峰，原因是 Cost 下滑速度比 Revenue 更快。**這正是「利潤最大化」不等於「營收最大化」的視覺證明。**


## 5. Assumption Validation：Unit Cost 與 Elasticity 合理嗎？

在往下做競品監測之前，先驗證 Phase 1 的兩個核心假設是否站得住腳（而不是憑空設定）。

**Unit Cost = 400 合理嗎？**

參考產業資料：10000mAh 行動電源的 B2B 代工出廠價約 **US$5.44–9.00**（約 NT$175–290），全球零售價一般落在 **US$22–34**（約 NT$700–1100），差異取決於芯片品質、PD 快充實作深度、認證等級等因素，不只是容量本身。

從「出廠價 NT$175–290」加上關稅、品牌認證（如 BSMI）、包裝客製化、品管、倉儲物流等成本，**Unit Cost = 400 是合理甚至偏保守的假設**。

**重要限制：不能假設競品的 Unit Cost 跟我們相同。** 競品（尤其是 Xiaomi 這類大型品牌）可能因為採購規模、垂直整合供應鏈，取得遠低於 400 的成本，因此能在較低價格下仍維持正利潤，而我們用自己的成本結構去複製同樣的低價，不一定划算。這是後續 Phase 5 策略模擬需要特別注意的地方：**「別人賣得動的價格，我們不一定賺得到錢」。**

**Elasticity = -1.8 合理嗎？**

產業研究顯示：一般標準消費性電子產品的彈性通常大於 1.3；品牌辨識度低、可替代性高的商品，彈性估計常在 2.0 以上。一個常見的類比是：品牌獨特、替代品有限的商品（如遊戲主機）彈性較低，而規格高度標準化、替代品眾多的商品（如一般規格筆電）彈性較高——這與我們的行動電源情境高度吻合。

**結論：-1.8 落在合理範圍內，甚至可以說偏保守**（考慮到行動電源比一般消費電子更加規格化，實際彈性可能更接近 -2.0 至 -2.5）。維持原假設，不做調整。


## 6. 結論與限制

**Key Findings**
- 在目前的假設參數下，估計的 profit-maximizing price 約為 **900**（相較目前 690 高出約 30%）。
- Revenue-maximizing 與 Profit-maximizing 並非同一個價格點，這對定價決策有直接意涵。
- Unit Cost 與 Elasticity 假設經產業數據交叉驗證，落在合理範圍內。

**Limitations**
- Demand、elasticity 皆為假設值，非真實銷售歷史資料估計而來。
- 模型假設彈性在整個價格區間內固定不變，實務上大幅價格變動時彈性可能改變。
- 目前的 demand model 為 own-price elasticity only，未納入競品價格作為輸入，故 Recommended Price 僅反映需求曲線本身的最適點，實務決策仍需搭配競爭情境模擬。
- 不能假設競品與我們有相同的 Unit Cost 結構（規模、供應鏈整合程度不同）。
